# 03 — Chatbot Inference Test
**CISC 886 — Cloud Computing**  
Tests the fine-tuned `cisc886-beauty-model.gguf` loaded directly from S3.  
Run with **T4 GPU** runtime.

**Cell order:**
1. Check GPU
2. Install dependencies
3. Configure AWS
4. Download model from S3
5. Load model
6. Test as chatbot (Section 5 qualitative comparison)
7. Compare base vs fine-tuned responses

In [1]:
# Cell 1 — Check GPU
import torch

print(f"GPU available:  {torch.cuda.is_available()}")
print(f"GPU name:       {torch.cuda.get_device_name(0)}")
print(f"GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA version:   {torch.version.cuda}")

GPU available:  True
GPU name:       Tesla T4
GPU memory:     15.6 GB
CUDA version:   12.8


In [2]:
# Cell 2 — Install dependencies
!pip install llama-cpp-python boto3 -q
print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.4 MB/s eta 0:00:00
✅ Dependencies installed


In [ ]:
# Cell 3 — Configure AWS credentials
import os
import boto3

os.environ['AWS_ACCESS_KEY_ID']     = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

BUCKET = '25tvtm-cisc886-bucket-cloud-project'

s3 = boto3.client(
    's3',
    aws_access_key_id     = os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key = os.environ['AWS_SECRET_ACCESS_KEY'],
    region_name           = 'us-east-1'
)

try:
    s3.head_bucket(Bucket=BUCKET)
    print(f"✅ Connected to S3 bucket: {BUCKET}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to S3 bucket: 25tvtm-cisc886-bucket-cloud-project


In [6]:
# Cell 4 — Download model from S3
MODEL_KEY  = 'model/cisc886-beauty-model.gguf'
LOCAL_PATH = '/content/cisc886-beauty-model.gguf'

total      = s3.head_object(Bucket=BUCKET, Key=MODEL_KEY)['ContentLength']
downloaded = [0]

def progress(chunk):
    downloaded[0] += chunk
    pct = downloaded[0] / total * 100
    print(f"\rDownloading... {pct:.1f}% ({downloaded[0]/1e9:.2f}/{total/1e9:.2f} GB)", end='')

print(f"Downloading from s3://{BUCKET}/{MODEL_KEY}")
s3.download_file(BUCKET, MODEL_KEY, LOCAL_PATH, Callback=progress)
print(f"\n✅ Downloaded: {LOCAL_PATH}")
print(f"   File size: {__import__('os').path.getsize(LOCAL_PATH)/1e9:.2f} GB")

Downloading... 100.0% (2.02/2.02 GB)
✅ Downloaded: /content/cisc886-beauty-model.gguf
   File size: 2.02 GB


In [7]:
# Cell 5 — Load model from GGUF
from llama_cpp import Llama

print("Loading model...")
llm = Llama(
    model_path   = LOCAL_PATH,
    n_ctx        = 512,
    n_gpu_layers = -1,    # use all GPU layers
    verbose      = False,
)
print("✅ Model loaded successfully")
print(f"   Model: cisc886-beauty-model.gguf")
print(f"   Base:  Llama-3.2-3B-Instruct (QLoRA fine-tuned)")
print(f"   Domain: Beauty & Personal Care reviews")

Loading model...


llama_context: n_ctx_seq (512) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ Model loaded successfully
   Model: cisc886-beauty-model.gguf
   Base:  Llama-3.2-3B-Instruct (QLoRA fine-tuned)
   Domain: Beauty & Personal Care reviews


In [12]:
def chat(user_message, max_tokens=300):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a knowledgeable beauty and personal care assistant. You help users find the right products, understand ingredients, compare options, and make informed purchasing decisions based on real customer experiences from Amazon reviews.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_message}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    output = llm(
        prompt,
        max_tokens     = max_tokens,
        temperature    = 0.7,
        repeat_penalty = 1.1,
        stop           = ["<|eot_id|>", "<|end_of_text|>"],  # removed "\n\n"
    )
    return output['choices'][0]['text'].strip()

In [13]:
# Cell 7 — Test chatbot with 5 beauty domain questions
# These demonstrate the model working as intended — a domain-specific assistant

test_questions = [
    "What should I look for in a good moisturizer for dry skin?",
    "Is CeraVe or Neutrogena better for sensitive skin?",
    "What are the most common complaints customers have about face serums?",
    "Can you recommend a good shampoo for damaged hair?",
    "What ingredients should I avoid in skincare products?",
]

print("=" * 65)
print("CISC886 BEAUTY ASSISTANT CHATBOT — FINE-TUNED MODEL TEST")
print("Model: cisc886-beauty-model.gguf (loaded from S3)")
print("=" * 65)

for i, question in enumerate(test_questions, 1):
    print(f"\nQ{i}: {question}")
    response = chat(question)
    print(f"A{i}: {response}")
    print("-" * 65)

print("\n✅ Chatbot inference test complete")

CISC886 BEAUTY ASSISTANT CHATBOT — FINE-TUNED MODEL TEST
Model: cisc886-beauty-model.gguf (loaded from S3)

Q1: What should I look for in a good moisturizer for dry skin?
A1: Choosing the right moisturizer for dry skin can be a bit overwhelming, but I'm here to help! For dry skin, you'll want to look for a moisturizer that's rich in emollients and humectants, which help lock in moisture. Here are some key ingredients to look for:

1. **Hyaluronic Acid**: A natural humectant that attracts and retains moisture.
2. **Glycerin**: Helps retain moisture and soothe dry skin.
3. **Ceramides**: Restores the skin's barrier function, preventing water loss and irritation.
4. **Niacinamide**: Improves skin elasticity and hydration.
5. **Occlusive ingredients** like petroleum jelly (Vaseline), dimethicone, or petrolatum help lock in moisture.

When shopping for a moisturizer, look for products labeled as:

* For dry skin
* Fragrance-free or hypoallergenic to minimize irritation
* Non-comedogenic (do

In [17]:
# Cell 8 — Proper base vs fine-tuned comparison
# Loads base Llama 3.2 3B from HuggingFace and compares to fine-tuned model

!pip install unsloth transformers -q





In [18]:
from unsloth import FastLanguageModel
import torch

print("Loading BASE model (no fine-tuning)...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 512,
    load_in_4bit   = True,
    dtype          = None,
)

FastLanguageModel.for_inference(base_model)
print("✅ Base model loaded")

def chat_base(user_message, max_tokens=200):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_message}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    inputs = base_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = 0.7,
            do_sample          = True,
            pad_token_id       = base_tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )
    return base_tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

# ── Comparison questions ───────────────────────────────────────────────────────
comparison_questions = [
    "What do customers commonly complain about when buying face moisturizers online?",
    "What makes a shampoo good for damaged hair based on customer reviews?",
]

print("\n" + "=" * 70)
print("SECTION 5 — BASE MODEL vs FINE-TUNED MODEL COMPARISON")
print("=" * 70)

for i, question in enumerate(comparison_questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTION {i}: {question}")
    print(f"{'='*70}")

    print(f"\n[ BASE MODEL — Llama 3.2 3B, no fine-tuning ]")
    base_response = chat_base(question)
    print(base_response)

    print(f"\n[ FINE-TUNED MODEL — trained on 50K Beauty & Personal Care reviews ]")
    finetuned_response = chat(question)
    print(finetuned_response)

Loading BASE model (no fine-tuning)...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Base model loaded

SECTION 5 — BASE MODEL vs FINE-TUNED MODEL COMPARISON

QUESTION 1: What do customers commonly complain about when buying face moisturizers online?

[ BASE MODEL — Llama 3.2 3B, no fine-tuning ]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Based on various reviews and feedback from customers, here are some common complaints people have when buying face moisturizers online:

1. **Not matching the product description**: Customers may feel that the actual product doesn't match what they saw in the pictures or description, either in terms of texture, color, or consistency.
2. **Quality and performance issues**: Some customers report that the product doesn't deliver on its promises, such as not providing sufficient hydration, not working well with their skin type, or not addressing specific concerns like acne or aging.
3. **Packaging and shipping**: Issues with packaging, such as broken or damaged products upon arrival, can be frustrating for customers who were expecting a high-quality product.
4. **Lack of transparency**: Some customers feel that companies don't provide clear information about ingredients, allergens, or potential side effects, making it difficult to make an informed decision.
5. **Unrealistic expectations**:

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1256: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on Amazon reviews, some common complaints customers have when buying face moisturizers online are:

1. Not as expected in terms of texture or consistency.
2. Too greasy or oily for their skin type.
3. Did not address specific skin concerns such as acne, aging, dryness, etc.
4. Had an unpleasant scent that did not suit them well.
5. Expired product despite the packaging saying it was good until the end of 2024.
6. Overpriced compared to similar products from other brands.
7. Packaging is fragile and may break during shipping.
8. Product didn't seem to work as expected for their skin concerns.
9. Not gentle enough for daily use without leaving residue.
10. Received a product that was not the size that was advertised.

QUESTION 2: What makes a shampoo good for damaged hair based on customer reviews?

[ BASE MODEL — Llama 3.2 3B, no fine-tuning ]
Based on various online reviews and ratings from customers, here are some key factors that make a shampoo good for damaged hair:

1. **Mois

In [21]:
your_question = "What are the best budget-friendly foundations recommended by customers?"

print(f"Question: {your_question}")
print(f"\nAnswer: {chat(your_question, max_tokens=200)}")

Question: What are the best budget-friendly foundations recommended by customers?

Answer: Based on Amazon reviews, here are some budget-friendly foundation recommendations from customers:

1. **Revlon Colorstay Foundation for Dry or Combination Skin**: 
    * Price: $10.99
    * Rating: 4.5/5 stars (over 2,600 reviews)
2. **Revlon ColorStay Foundation for Normal to Dry Skin**: 
    * Price: $11.48
    * Rating: 4.4/5 stars (over 1,500 reviews)
3. **Maybelline Fit Me! Dewy + Smooth Foundation**: 
    * Price: $10.99
    * Rating: 4.6/5 stars (over 2,100 reviews)
4. **L'Oreal Paris True Match Super Stay Foundation for Normal to Dry Skin**: 
    * Price: $11.98
    * Rating: 4.3/5 stars (over 2,200 reviews)
5. **Maybelline Fit
